# PSFSim vs. STPSF

In [ ]:
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import stpsf
from matplotlib.colors import LinearSegmentedColormap, LogNorm, SymLogNorm
from numpy.fft import fft2, fftfreq, ifft2
from scipy.optimize import minimize
from stpsf import roman
from psfsim.polychrom import PolychromaticPSF, inBandpass
from psfsim.wfi_coordinate_transformations import from_fpa_to_angle, from_sca_to_fpa

# ---------------------------------------------------------------- what to compare
SCA = 7  # SCA number, 1-18
X_SCI, Y_SCI = 2044, 2044
FILTER, STPSF_FILTER = "H", "F158"  # matching filter in each code
CYCLE = 10

WAVELENGTHS = np.linspace(1.38, 1.77, 8)  # microns
OVSAMP = 10  # samples per native pixel
PUPIL = 192  # use_postage_stamp_size: sets the FFT size, 192*10 = 1920
STAMP = 96  # native pixels actually compared
STPSF_EXT = "OVERSAMP"  # optics only, to match optical_psf_only=True

In [ ]:
def centroid(im):
    y, x = np.mgrid[0 : im.shape[0], 0 : im.shape[1]]
    w = np.clip(im, 0, None)
    s = w.sum()
    return (x * w).sum() / s, (y * w).sum() / s


def fshift(im, dx, dy):
    # sub-pixel translation by (dx, dy) with a Fourier phase ramp
    ny, nx = im.shape
    fy, fx = fftfreq(ny)[:, None], fftfreq(nx)[None, :]
    return np.real(ifft2(fft2(im) * np.exp(-2j * np.pi * (fx * dx + fy * dy))))


def xcorr_shift(ref, im):
    # integer shift that maximises the cross-correlation of im with ref
    c = np.real(ifft2(fft2(ref) * np.conj(fft2(im))))
    iy, ix = np.unravel_index(np.argmax(c), c.shape)
    ny, nx = c.shape
    return (ix - nx if ix > nx // 2 else ix), (iy - ny if iy > ny // 2 else iy)


def align(ref, im):
    """Shift im onto ref by the sub-pixel shift that maximises their correlation.

    NB not centroid matching: the two codes genuinely disagree about the first
    moment, so forcing their centroids together misaligns the cores.
    """
    r = minimize(
        lambda p: -corr(ref, fshift(im, *p)),
        x0=xcorr_shift(ref, im),
        method="Nelder-Mead",
        options=dict(xatol=1e-3, fatol=1e-10),
    )
    out = fshift(im, *r.x)
    return out / out.sum(), r.x


def crop(a, n):
    o = (a.shape[0] - n) // 2
    return a[o : o + n, o : o + n]


def corr(a, b):
    return float(np.sum(a * b) / np.sqrt(np.sum(a * a) * np.sum(b * b)))

## 1. Compute the two PSFs

In [ ]:
poly = PolychromaticPSF(SCA, X_SCI, Y_SCI, WAVELENGTHS, frame="fpa")

In [ ]:
psfsim_psf = poly.compute_poly_psf(
    postage_stamp_size=STAMP,
    optical_psf_only=True,
    use_postage_stamp_size=PUPIL,
    ovsamp=OVSAMP,
    use_filter=FILTER,
    cycle=CYCLE,
)
print("PSFSim ", psfsim_psf.shape, f" (= {PUPIL} x {OVSAMP})")
for w in WAVELENGTHS:
    print(f"   {w:.4f} um  in band: {inBandpass(float(w), FILTER, poly.bandpass)[0]}")

In [ ]:
wfi = roman.WFI()
wfi.filter = STPSF_FILTER
wfi.detector = f"SCA{SCA:02d}"
wfi.detector_position = (X_SCI, Y_SCI)
# PSFSim's own quadrature weights, so STPSF is combined in exactly the same way
wl = np.sort(WAVELENGTHS)
tw = np.empty_like(wl)
tw[0] = 0.5 * (wl[1] - wl[0])
tw[-1] = 0.5 * (wl[-1] - wl[-2])
tw[1:-1] = 0.5 * (wl[2:] - wl[:-2])

stpsf_psf = None
for wav, weight in zip(wl, tw, strict=True):
    mono = wfi.calc_psf(monochromatic=wav * 1e-6, oversample=OVSAMP, fov_pixels=STAMP)[STPSF_EXT].data
    stpsf_psf = weight * mono if stpsf_psf is None else stpsf_psf + weight * mono
stpsf_psf /= stpsf_psf.sum()

In [ ]:
n = stpsf_psf.shape[0]
inside = crop(psfsim_psf, n).sum()

A = crop(psfsim_psf, n) / inside  # PSFSim, as delivered
B = stpsf_psf  # STPSF,  as delivered
B_a, shift = align(A, B)  # STPSF shifted onto PSFSim, best-correlation shift
cen_a, cen_b = centroid(A), centroid(B)
print(
    f"comparing {n} x {n} oversampled pixels ({STAMP} native x {OVSAMP});"
    f" {100 * inside:.1f}% of PSFSim's padded array is inside"
)
print(f"centroid  PSFSim  ({cen_a[0]:.3f}, {cen_a[1]:.3f})")
print(f"          STPSF   ({cen_b[0]:.3f}, {cen_b[1]:.3f})")
print(
    f"offset            ({cen_b[0] - cen_a[0]:+.3f}, {cen_b[1] - cen_a[1]:+.3f})"
    f" oversampled pixels"
    f"  = ({(cen_b[0] - cen_a[0]) / OVSAMP:+.3f}, {(cen_b[1] - cen_a[1]) / OVSAMP:+.3f}) native"
)
print(
    f"alignment shift applied to STPSF  ({shift[0]:+.3f}, {shift[1]:+.3f}) oversampled"
    f"  = ({shift[0] / OVSAMP:+.3f}, {shift[1] / OVSAMP:+.3f}) native"
)

## 2. The two PSFs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.6), constrained_layout=True)
for ax, im, name in zip(axes, (A, B), ("PSFSim", "STPSF"), strict=True):
    m = ax.imshow(im, origin="lower", vmax=1e-6)
    ax.set_title(name)
    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
cbar = fig.colorbar(m, ax=axes, fraction=0.045, pad=0.02)
cbar.set_label("fraction of total flux per oversampled pixel")
cbar.outline.set_visible(False)
fig.suptitle(
    f"SCA{SCA:02d}  {poly.frame}({X_SCI}, {Y_SCI})   "
    f"{WAVELENGTHS[0]:.2f}-{WAVELENGTHS[-1]:.2f} um in {len(WAVELENGTHS)} nodes   "
    f"linear, clipped at {m.get_clim()[1]:.0e}",
    x=0.01,
    ha="left",
    fontsize=10,
)
plt.show()

## 3. Difference

Left: the arrays exactly as each code delivers them. Right: STPSF shifted onto PSFSim by
the sub-pixel shift that maximises their correlation. Both panels share one symmetric-log
colour scale — on a linear scale the core swamps everything else.

Registration is by cross-correlation, *not* by matching flux centroids. The two codes
disagree about the first moment of the PSF, so forcing their centroids together over-shifts
and pulls the cores apart — here by about 2 oversampled pixels in y, which is enough to
dominate the residual.

In [ ]:
panels = {"no alignment": (A, B), "aligned on the core": (A, B_a)}
resid = {k: a - b for k, (a, b) in panels.items()}
lim = max(np.abs(d).max() for d in resid.values())
cut = 420

fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.9), constrained_layout=True)
for ax, (name, (a, b)) in zip(axes, panels.items(), strict=True):
    d = resid[name]
    m = ax.imshow(d[cut:-cut, cut:-cut], origin="lower", vmin=-lim, vmax=lim)
    ax.set_title(name)
    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    ax.text(
        0.03,
        0.03,
        f"rms resid / peak  {np.std(d) / a.max():.4f}\ncorrelation  {corr(a, b):+.4f}",
        transform=ax.transAxes,
        va="bottom",
        ha="left",
        fontsize=8,
    )
cbar = fig.colorbar(m, ax=axes, fraction=0.045, pad=0.02)
cbar.set_label("PSFSim  -  STPSF   (fraction of total flux)")
cbar.outline.set_visible(False)
fig.suptitle("PSFSim minus STPSF", x=0.01, ha="left", fontsize=10)
plt.show()

In [ ]:
print(f"{'':22s} {'corr':>8s} {'rms/peak':>10s} {'max|diff|/peak':>16s}")
print("-" * 60)
for name, (a, b) in panels.items():
    d = resid[name]
    print(f"{name:22s} {corr(a, b):+8.4f} {np.std(d) / a.max():10.4f}" f" {np.abs(d).max() / a.max():16.4f}")